In [ ]:
import numpy as np              # NumPy数值计算库
import matplotlib.pyplot as plt  # Matplotlib绑图库
import cv2 as cv                 # OpenCV计算机视觉库

In [ ]:
def show(img):
    """自定义显示函数：自动判断灰度图或彩色图并正确显示"""
    if img.ndim == 2:  # 灰度图
        plt.imshow(img, cmap='gray')
    else:  # 彩色图，BGR转RGB
        plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))
    plt.show()

## part2. 图像的线性和非线性变换

In [ ]:
# 读取灰度图用于线性和非线性变换演示
img = cv.imread('pic/cat500x480.jpg', 0)
show(img)

### 3.4线性变化

- 手写线性变化

In [ ]:
# 手动线性变换：s = b + k*r，但uint8类型下乘以k后超过255会取模溢出
b = 20
k = 2
img2 = b + k * img

show(img2)
print(img2)

因为img是uint8类型的，乘以k之后会超过255，他会循环为一些奇怪的数字

In [ ]:
# 转为int32避免溢出，但超过255的值会保留（可能大于255）
b = 20
k = 2
img2 = b + k * img.astype(np.int32)
show(img2)
print(img2.max())  # 最大值可能超过255

In [ ]:
# 正确做法：转为int32后用np.clip截断到[0,255]
b = 20
k = 2
img2 = b + k * img.astype(np.int32)
img2 = np.clip(img2, 0, 255)  # 截断：小于0设为0，大于255设为255
show(img2)
print(img2.max())  # 最大值为255

- convertScaleAbs函数

In [ ]:
# cv.convertScaleAbs()：OpenCV内置线性变换函数，alpha=斜率k，beta=截距b
# 自动处理溢出并返回uint8类型
img3 = cv.convertScaleAbs(img, alpha=2, beta=20)
show(img3)

### 3.5对数变换

In [ ]:
# 对数变换：s = a + ln(r+1)/b，+1避免log(0)
# 注意：直接对uint8做log会溢出，这里会有问题
img4 = 10 + np.log(img + 1) / 0.1
show(img4)

In [9]:
np.uint8([-1, 254, 255])

OverflowError: Python integer -1 out of bounds for uint8

In [ ]:
np.uint8([-1, 254, 255]) + 1

OverflowError: Python integer -1 out of bounds for uint8

In [ ]:
# 正确的对数变换：先转为float32，避免uint8溢出
img4 = 10 + np.log(img.astype(np.float32) + 1) / 0.1
show(img4)

### 3.6伽马变换

In [ ]:
# 伽马变换（幂律变换）：s = c * r^gamma
# gamma<1（如0.5）提亮暗部；gamma>1（如1.5）压暗亮部
img01 = img / 255                              # 归一化到[0,1]
img05 = np.power(img01, 0.5) * 255            # gamma=0.5
img15 = np.power(img01, 1.5) * 255            # gamma=1.5

show(np.hstack([img05, img, img15]))  # 并排显示：gamma<1 | 原图 | gamma>1

### 3.7阈值变换

常用的阈值分割方法有以下五种：
- 1. cv2.THRESH_BINARY：黑白二值，高于阈值的归1，低于的归0；
- 2. cv2.THRESH_BINARY _INV：黑白二值翻转，第一种情况的反作用；
- 3. cv2.THRESH_TRUNC：把大于阈值的变成等于阈值；
- 4. cv2.THRESH_TOZERO：当像素高于阈值时像素设置为自己设置的像素值，低于阈值时不作处理；
- 5. cv2.THRESH_TOZERO_INV：当像素低于阈值时设置为自己设置的像素值，高于阈值时不作处理。

In [ ]:
# 重新读取灰度图用于阈值变换演示
img = cv.imread('pic/cat500x480.jpg', 0)
show(img)

In [ ]:
# cv.threshold()：五种阈值分割方法比较
# 参数：(灰度图, 阈值127, 最大值255, 分割类型)
ret, thresh1 = cv.threshold(img, 127, 255, cv.THRESH_BINARY)       # 二值化：>127设为255，<=127设为0
ret, thresh2 = cv.threshold(img, 127, 255, cv.THRESH_BINARY_INV)   # 反二值化：>127设为0，<=127设为255
ret, thresh3 = cv.threshold(img, 127, 255, cv.THRESH_TRUNC)        # 截断：>127截断为127，<=127不变
ret, thresh4 = cv.threshold(img, 127, 255, cv.THRESH_TOZERO)       # 低于阈值归零，高于不变
ret, thresh5 = cv.threshold(img, 127, 255, cv.THRESH_TOZERO_INV)   # 高于阈值归零，低于不变

In [ ]:
# 并排显示原图和五种阈值分割结果
show(np.hstack([img, thresh1, thresh2, thresh3, thresh4, thresh5]))